In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

from sklearn.metrics import confusion_matrix, classification_report

In [3]:
data_dir = "/content/drive/MyDrive/전심프/data/hairloss_forehead_split"

batch_size = 16
num_workers = 2

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.1
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

test_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(
    root=data_dir + "/train",
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    root=data_dir + "/val",
    transform=test_transform
)

test_dataset = datasets.ImageFolder(
    root=data_dir + "/test",
    transform=test_transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)

print("Classes:", train_dataset.classes)
print("Class to idx:", train_dataset.class_to_idx)
print("Train:", len(train_dataset))
print("Val:", len(val_dataset))
print("Test:", len(test_dataset))

Classes: ['1_level', '3_level']
Class to idx: {'1_level': 0, '3_level': 1}
Train: 513
Val: 109
Test: 113


In [4]:
def create_model(num_classes=2, pretrained=True):
    """
    EfficientNet-B0 기반 이마 탈모 2-class 분류 모델 생성
    1_level: 정상 또는 약한 단계
    3_level: 탈모 의심 단계
    """

    if pretrained:
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
    else:
        weights = None

    model = models.efficientnet_b0(weights=weights)

    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)

    return model

In [5]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = create_model(num_classes=2, pretrained=True)
model = model.to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.0001
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

num_epochs = 30
best_val_acc = 0.0

early_stop_patience = 7
early_stop_count = 0

save_path = "/content/drive/MyDrive/전심프/best_model_forehead.pth"

for epoch in range(num_epochs):
    model.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    train_acc = 100 * train_correct / train_total

    model.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            val_total += labels.size(0)
            val_correct += (predicted == labels).sum().item()

    val_acc = 100 * val_correct / val_total

    scheduler.step(val_acc)

    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Epoch [{epoch+1}/{num_epochs}] "
        f"Train Loss: {train_loss / len(train_loader):.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Loss: {val_loss / len(val_loader):.4f} "
        f"Val Acc: {val_acc:.2f}% "
        f"LR: {current_lr:.6f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        early_stop_count = 0
        torch.save(model.state_dict(), save_path)
        print("Best model saved!", save_path)
    else:
        early_stop_count += 1
        print(f"Early stopping count: {early_stop_count}/{early_stop_patience}")

    if early_stop_count >= early_stop_patience:
        print("Early stopping triggered.")
        break

print("Training finished.")
print(f"Best Validation Accuracy: {best_val_acc:.2f}%")

Device: cuda
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 232MB/s]


Epoch [1/30] Train Loss: 0.6155 Train Acc: 65.69% Val Loss: 0.4621 Val Acc: 80.73% LR: 0.000100
Best model saved! /content/drive/MyDrive/전심프/best_model_forehead.pth
Epoch [2/30] Train Loss: 0.4657 Train Acc: 81.48% Val Loss: 0.4157 Val Acc: 80.73% LR: 0.000100
Early stopping count: 1/7
Epoch [3/30] Train Loss: 0.3767 Train Acc: 85.19% Val Loss: 0.3893 Val Acc: 78.90% LR: 0.000100
Early stopping count: 2/7
Epoch [4/30] Train Loss: 0.3063 Train Acc: 87.13% Val Loss: 0.4009 Val Acc: 79.82% LR: 0.000100
Early stopping count: 3/7
Epoch [5/30] Train Loss: 0.2435 Train Acc: 90.45% Val Loss: 0.3357 Val Acc: 82.57% LR: 0.000100
Best model saved! /content/drive/MyDrive/전심프/best_model_forehead.pth
Epoch [6/30] Train Loss: 0.2266 Train Acc: 90.64% Val Loss: 0.3418 Val Acc: 83.49% LR: 0.000100
Best model saved! /content/drive/MyDrive/전심프/best_model_forehead.pth
Epoch [7/30] Train Loss: 0.1875 Train Acc: 92.98% Val Loss: 0.4714 Val Acc: 80.73% LR: 0.000100
Early stopping count: 1/7
Epoch [8/30] Trai

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_path = "/content/drive/MyDrive/전심프/best_model_forehead.pth"

model = create_model(num_classes=2, pretrained=False)

model.load_state_dict(
    torch.load(
        model_path,
        map_location=device,
        weights_only=True
    )
)

model = model.to(device)
model.eval()

correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = 100 * correct / total

print(f"\nTest Accuracy: {accuracy:.2f}%")

cm = confusion_matrix(all_labels, all_preds)

print("\nConfusion Matrix")
print(cm)

print("\nClassification Report")
print(
    classification_report(
        all_labels,
        all_preds,
        target_names=test_dataset.classes,
        digits=4
    )
)


Test Accuracy: 86.73%

Confusion Matrix
[[41  8]
 [ 7 57]]

Classification Report
              precision    recall  f1-score   support

     1_level     0.8542    0.8367    0.8454        49
     3_level     0.8769    0.8906    0.8837        64

    accuracy                         0.8673       113
   macro avg     0.8655    0.8637    0.8645       113
weighted avg     0.8671    0.8673    0.8671       113



In [8]:
import torch
import torch.nn.functional as F
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class_names = train_dataset.classes

model_path = "/content/drive/MyDrive/전심프/best_model_forehead.pth"

model = create_model(num_classes=2, pretrained=False)
model.load_state_dict(
    torch.load(model_path, map_location=device, weights_only=True)
)
model = model.to(device)
model.eval()


def predict_image(image_path):
    image = Image.open(image_path).convert("RGB")

    input_tensor = test_transform(image)
    input_tensor = input_tensor.unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(input_tensor)
        probs = F.softmax(outputs, dim=1)[0]

    pred_idx = torch.argmax(probs).item()
    pred_class = class_names[pred_idx]

    print("예측 결과:", pred_class)
    print("\n클래스별 확률")

    for class_name, prob in zip(class_names, probs):
        print(f"{class_name}: {prob.item() * 100:.2f}%")

    return pred_class, probs

In [26]:
image_path = "/content/drive/MyDrive/전심프/realtestdata/test5.png"

predict_image(image_path)

예측 결과: 1_level

클래스별 확률
1_level: 98.93%
3_level: 1.07%


('1_level', tensor([0.9893, 0.0107], device='cuda:0'))